# Milestone 1: Maximizing Inference with Majority Voting

## 1. Configuration and Setup

This cell contains all the configuration variables. We define the model, data paths, and a new output path for our aggregated results.

In [ ]:
import json
import os
import re
from collections import Counter
from typing import List, Optional

from tqdm import tqdm
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

# ─── Configuration ─────────────────────────────────────────────────
MODEL_ID = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID = "0"
DATA_PATH = "data/public.jsonl"
OUTPUT_PATH = "results/milestone1_results.jsonl" # New output file for this strategy

# Set the device environment variable
os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID

## 2. Data Loading and Prompt Engineering

We load the dataset and define the prompt-building functions exactly as in the baseline. This part remains unchanged.

In [ ]:
data = [json.loads(line) for line in open(DATA_PATH)]
print(f"Loaded {len(data)} questions.")

SYSTEM_PROMPT_MATH = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

SYSTEM_PROMPT_MCQ = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    """Return (system_prompt, user_prompt) for a question."""
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    return SYSTEM_PROMPT_MATH, question

## 3. Model and Sampling Configuration

Here we initialize the model and tokenizer. The key change is in `SamplingParams`:
1.  `n=5`: We instruct vLLM to generate 5 different output sequences for each prompt.
2.  `temperature=0.7`: We use a non-zero temperature to encourage diversity in the generations, which is essential for majority voting to be effective.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

llm = LLM(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=False,
    gpu_memory_utilization=0.50,
    max_model_len=8192,
    trust_remote_code=True,
    max_num_seqs=1, # Process one prompt at a time
    max_num_batched_tokens=32768,
)

# **ACTION**: Modify SamplingParams to generate multiple outputs.
sampling_params = SamplingParams(
    n=5,                  # Generate 5 different outputs.
    temperature=0.7,      # Use temperature to get diverse generations.
    top_p=0.95,
    max_tokens=4096,      # Max tokens per generation.
    use_beam_search=False # Must be False for n > 1 with temperature sampling.
)

print("Model and new sampling parameters loaded.")

## 4. Aggregation Logic

This function takes a list of generated responses, extracts the `\boxed{}` answer from each, and returns the most common answer. This is the majority voting mechanism.

In [ ]:
def majority_vote_boxed_answer(responses: List[str]) -> str:
    """
    Extracts the last \boxed{} answer from a list of LLM responses and 
    returns the most common answer (majority vote).

    Args:
        responses: A list of string outputs from the language model.

    Returns:
        The most frequently occurring answer. If no boxed answers are found,
        it returns the full text of the first response as a fallback.
    """
    boxed_answers = []
    pattern = re.compile(r"\\boxed{(.*?)}", re.DOTALL)
    
    for response in responses:
        matches = pattern.findall(response)
        if matches:
            # Take the last boxed answer as the final one for this generation
            last_answer = matches[-1].strip()
            boxed_answers.append(last_answer)

    # If no boxed answers were found in any generation, return the first full response
    if not boxed_answers:
        return responses[0] if responses else ""

    # Use Counter to find the most common answer
    vote_counts = Counter(boxed_answers)
    # most_common(1) returns a list like [('answer', count)]
    most_common_answer = vote_counts.most_common(1)[0][0]
    
    return most_common_answer

## 5. Generation and Submission

We now loop through the entire dataset. For each question, we generate 5 responses, aggregate them using our majority vote function, and store the final result. Finally, we save the results to a `.jsonl` file in the required submission format.

In [ ]:
submission_records = []

print(f"Generating responses for {len(data)} questions...")

# Use tqdm for a progress bar
for item in tqdm(data, desc="Processing questions"):
    # 1. Build the prompt
    system, user = build_prompt(item["question"], item.get("options"))
    prompt_text = tokenizer.apply_chat_template(
        [{"role": "system", "content": system},
         {"role": "user",   "content": user}],
        tokenize=False,
        add_generation_prompt=True,
    )
    
    # 2. Generate multiple outputs for the single prompt
    # llm.generate expects a list of prompts
    vllm_outputs = llm.generate([prompt_text], sampling_params, use_tqdm=False)
    
    # 3. Extract the text from each of the 'n' generations
    # vllm_outputs is a list containing one RequestOutput object
    generated_responses = [output.text.strip() for output in vllm_outputs[0].outputs]
    
    # 4. Aggregate the answers using majority voting
    final_answer = majority_vote_boxed_answer(generated_responses)
    
    # 5. Store the result for submission
    submission_records.append({
        "id": item.get("id"),
        "response": final_answer
    })

print("\nGeneration complete.")

## 6. Save Results

Save the aggregated responses to the output file.

In [ ]:
from pathlib import Path

out_path = Path(OUTPUT_PATH)
out_path.parent.mkdir(parents=True, exist_ok=True)

with open(out_path, "w") as f:
    for record in submission_records:
        f.write(json.dumps(record) + "\n")

print(f"Saved {len(submission_records)} records to {out_path}")